In [16]:
import numpy as np

In [1]:
from opty import Problem, create_objective_function, parse_free
import sympy as sp
import numpy as np
import scipy as sc
import time as tm
import pickle
import sympy.physics.mechanics as me
import sys
sys.path.insert(0, "..")
from importlib import reload
import matplotlib.pyplot as plt
import equations as eq
reload (eq);
import trajectory_lib as tr
reload (tr);

participant = ['3NA','8tata']
motion_list  =  ['Elevation_yzy','Scabduction_yzy','Flexion_yzy'] #

GH_seq = 'YZY' 

act_w = 1
vel_w = 1

In [2]:
clav_pos = 0.4
tilt_y = 13
tilt_z = -6.5
weight = 200
quat_constraints = [0.15,0.15,0.15,0.15,0.15,0.15,0.15,0.15,0.15,0.15]
w_optim_fmax = 1
w_optim_lceopt = 1

for ipar in range(len(participant)):
    for imot in range(len(motion_list)):
        OS_struct = sc.io.loadmat('../Motions/'+participant[ipar]+'/OS_model.mat')
        MM,FO,q,u,fr,frstar,kinematical,xdot,first_elips_scale,elips_trans = eq.create_eoms_quat_no_RF(OS_struct,weight = 0,derive = 'numeric',gen_matlab_functions = 0)

        iweight = 10 #weights[isim]
        w_inf = 0
        # tilt_z = tilt_zs[isim]
        # iweight = 9
        wGH = 0.4

        TE,activations,TE_conoid, fmax_init, fmax_range, lceopt_init, lceopt_range, mus_groups, GH_mus_forces, mus_forces_objective = eq.polynomials_quat(OS_struct,q,u,calibrated_params = None, derive = 'numeric', RC_lim = 1.0)
        num_params = 0
        include_activation_dynamics = True
        optimize = 0


        # weight = 52
        struct_name = 'res_quat_'+motion_list[imot]+'_'+str(int(weight))
        eoms_implicit = sp.Matrix(kinematical).col_join(fr+frstar+sp.Matrix([TE+sp.Matrix(TE_conoid)]))
        traj_w = weight

        if include_activation_dynamics:
            excitations = []
            act_ode = []
            for i in range(len(activations)):
                excitations.append(me.dynamicsymbols('exc'+str(activations[i])[3:-3]))
                current_mus_ind = int(str(activations[i])[4:-3])
                current_mus = OS_struct['model']['muscles'].item()[0,(current_mus_ind-1)]
                t_act = current_mus['tact'][0,0].item()
                t_deact = current_mus['tdeact'][0,0].item()
                act_ode.append(activations[i].diff() - eq.act_dynamics(activations[i],excitations[i],t_act,t_deact))
            # print('act_ode = ', act_ode)
            sp_act_ode = sp.Matrix(act_ode)
            eoms_implicit = eoms_implicit.col_join(sp_act_ode)

        interval_value = 0.04
        file = '../Motions/' + participant[ipar] + '/' + motion_list[imot] + '/' + motion_list[imot]
        traj_original, omega, num_nodes, time = tr.exp_trajectory_quat(file,interval_value)
        q0_t0 = traj_original[:,0][:4]
        indexes = np.ones(num_nodes)
        traj = tr.exp_trajectory_quat_myobj(traj_original,clav_pos)
        indexes_clav_scap = 1
        indexes_hum = 1

        if include_activation_dynamics:
            state_symbols = tuple(q+u+activations)
            specified_symbols = tuple(excitations)
        else:
            state_symbols = tuple(q+u)
            specified_symbols = tuple(activations)

        num_states = len(state_symbols) 
        num_q = len(q)
        num_u = len(u)
        num_faux = 0
        num_inputs = len(specified_symbols)
        t = me.dynamicsymbols._t
        
        objective_traj,objective_traj_jac, objective_SC_t0, objective_SC_t0_jac = eq.custom_objective_quat(num_q,interval_value,clav_pos,True)

        objective_act,objective_act_jac = eq.min_activation(activations,interval_value)
        objective_exc,objective_exc_jac = eq.min_activation(activations,interval_value)

        obj_min_diff,obj_min_diff_jac = eq.objective_activation_diff(num_nodes,interval_value)

        
        w_diff_exc = 1e-3
        w_diff_vel = 1e-3


        def obj(free):
            min_traj = traj_w * np.sum(objective_traj(np.split(free[:num_q*num_nodes],num_q),traj,indexes_clav_scap,indexes_hum))
            min_SC_t0 = traj_w * np.sum(objective_SC_t0(free[0::num_nodes][:4],q0_t0))

            min_vel_dif = w_diff_vel * np.sum((obj_min_diff(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))))

            min_torque = act_w * np.sum(objective_act(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes],num_inputs)))
           

            obj = (min_traj + min_vel_dif + min_torque + min_SC_t0) #   
            if include_activation_dynamics:
                min_exc_dif = w_diff_exc * np.sum((obj_min_diff(np.transpose(np.split(free[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes],num_inputs)))))
                obj += (min_exc_dif)

            return obj.item()

        def obj_grad(free):
            grad = np.zeros_like(free)
            grad[:num_q*num_nodes] += traj_w * np.concatenate(objective_traj_jac(np.split(free[:num_q*num_nodes],num_q),traj,indexes_clav_scap,indexes_hum))
            grad[0::num_nodes][:4] += traj_w * np.sum(objective_SC_t0_jac(free[0::num_nodes][:4],q0_t0))

            grad[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes] += act_w * np.concatenate(objective_act_jac(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes],num_inputs)))

            grad[num_q*num_nodes:(num_q + num_u)*num_nodes] += w_diff_vel * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))[0,:,:])))


            if include_activation_dynamics:
                grad[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes] += w_diff_exc * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes],num_inputs)))[0,:,:])))

            return grad

        print('obj_check', obj(np.ones(num_states*num_nodes + num_inputs*num_nodes + num_params)*0.01))
        print('obj_grad_check', sum(obj_grad(np.ones(num_states*num_nodes + num_inputs*num_nodes + num_params)*0.01)))

        instance_constraints = []
        instance_constraints.append(state_symbols[0].func(0)**2 + state_symbols[1].func(0)**2 + state_symbols[2].func(0)**2 + state_symbols[3].func(0)**2 - 1) # SC
        instance_constraints.append(state_symbols[4].func(0)**2 + state_symbols[5].func(0)**2 + state_symbols[6].func(0)**2 + state_symbols[7].func(0)**2 - 1) # AC
        instance_constraints.append(state_symbols[8].func(0)**2 + state_symbols[9].func(0)**2 + state_symbols[10].func(0)**2 + state_symbols[11].func(0)**2 - 1) # GH

            
        bounds1 = (0.0,1.0)
        bounds = (bounds1,)*len(activations)
        bndrs = dict(zip(activations,bounds))
        if include_activation_dynamics:
            bndrs_exc = dict(zip(excitations,(bounds1,)*len(excitations)))
            bndrs.update(bndrs_exc)
        
        for i in range(num_q):
            bndrs.update({q[i]: (min(traj_original[i,:])-0.15, max(traj_original[i,:])+0.15)})

        print(bndrs)

        start = tm.time()
        prob = Problem(obj, obj_grad, eoms_implicit, state_symbols,
                    num_nodes, interval_value,
                    known_parameter_map={},
                    instance_constraints=instance_constraints,
                    bounds=bndrs,
                    integration_method='midpoint',
                    parallel = False)


        time_to_create = tm.time() - start
        print(time_to_create)

        prob.add_option('limited_memory_max_history', 40)
        initial_guess = np.ones(prob.num_free)*0.0

        time_2_solve_start = tm.time()

        prob.add_option('max_iter',2000)
        initial_guess[:13*num_nodes] = traj_original.flatten()
        initial_guess[num_q * num_nodes : (num_q + num_u) * num_nodes] = omega.flatten()
  
        solution, info = prob.solve(initial_guess)
        time_2_solve = tm.time() - time_2_solve_start
        print(info['status_msg'])
        print(info['obj_val'])
        act_obj = np.sum(solution[num_states*num_nodes:(num_states + num_inputs)*num_nodes]**2)
        objective_value = prob.obj_value
        print('Objective activations: ', act_obj)

        file_name = '../Motions/'+participant[ipar]+'/'+motion_list[imot]+'/' + struct_name+'.mat'

        tr.sol2struct(solution,activations,num_q,num_u,num_faux,num_inputs,num_nodes,time,objective_value,time_2_solve,file_name,include_activation_dynamics)

        file_name_mot = '../Motions/'+participant[ipar]+'/'+motion_list[imot]+'/' + struct_name + '.mot'
        tr.sol2mot_quat(solution, num_nodes, len(q), time, file_name_mot, GH_seq)

[0.23231129464231123, 1.0504640207736304, 3.0556381120292206, 0.8136852012343517, 0.8136852012343517, 0.7818168569693167]
infra1  .. 272.0
infra2  .. 275.0
infra3  .. 194.0
infra4  .. 207.0
infra5  .. 302.0
infra6  .. 182.0
supra1  .. 120.0
supra2  .. 113.0
supra3  .. 258.0
supra4  .. 130.0
[0.1280213209318535, 0.09977871970276542, 0.09473223609756792, 0.08428512968680808, 0.0818061552842549, 0.0790615764814282, 0.07684820647914856, 0.06817179607021243, 0.089508682892188, 0.11190798731525782, 0.11996465412355564, 0.13696333574106312, 0.14873846415319078, 0.1397079145438899, 0.11527230971872285, 0.06772912206975651, 0.09614879289902686, 0.09517491009802384, 0.12226655892592643, 0.11385575291726387, 0.12191241972556167, 0.1380257533421574, 0.13873403174288687, 0.13899963614316038, 0.09738822274168242, 0.11748559310816087, 0.11704286818258923, 0.11412124455528062, 0.11146519949003068, 0.1080123389766735, 0.1003097998889621, 0.07481180430570758, 0.07082777389651698, 0.05843292559889128, 0.

KeyboardInterrupt: 

In [19]:
from opty import Problem, create_objective_function, parse_free
import sympy as sp
import numpy as np
import scipy as sc
import time as tm
import pickle
import sympy.physics.mechanics as me
import sys
import matplotlib.pyplot as plt
sys.path.insert(0, "..")
from importlib import reload

import equations as eq
import trajectory_lib as tr
reload (tr);
reload (eq);

participant = ['3NA','8tata']
motion_list  =  ['Elevation_yzy','Scabduction_yzy','Flexion_yzy'] #
GH_seq = 'YZY' 
weight= 50

include_activation_dynamics = True

act_w = 1


In [20]:
for ipar in range(len(participant)):
    for imot in range(len(motion_list)):
        OS_struct = sc.io.loadmat('../Motions/'+participant[ipar]+'/OS_model.mat')
        MM,FO,q,u,fr,frstar,kindeq,xdot,first_elips_scale,elips_trans = eq.create_eoms_eul(OS_struct,derive = 'numeric',gen_matlab_functions = 0,GH_seq = GH_seq)
        TE,activations,TE_conoid = eq.polynomials_euler(OS_struct,q,u,derive = 'numeric')

        traj_w = weight
        struct_name = 'res_euler_'+motion_list[imot]+'_'+str(weight)
        eoms_implicit = sp.Matrix(kindeq).col_join(fr+frstar+TE+sp.Matrix(TE_conoid))

        if include_activation_dynamics:
            excitations = []
            act_ode = []
            # print(activations)
            for i in range(len(activations)):
                excitations.append(me.dynamicsymbols('exc'+str(activations[i])[3:-3]))
                current_mus_ind = int(str(activations[i])[4:-3])
                current_mus = OS_struct['model']['muscles'].item()[0,(current_mus_ind-1)]
                t_act = current_mus['tact'][0,0].item()
                t_deact = current_mus['tdeact'][0,0].item()
                act_ode.append(activations[i].diff() - eq.act_dynamics(activations[i],excitations[i],t_act,t_deact))

            sp_act_ode = sp.Matrix(act_ode)
            eoms_implicit = eoms_implicit.col_join(sp_act_ode)
        reload(eq);
        # num_nodes = 101
        file = '../Motions/'+participant[ipar]+'/'+motion_list[imot]+'/'+motion_list[imot]

        interval_value = 0.04
        traj_original, velocity, num_nodes, time = tr.exp_trajectory_eul(file,interval_value)
        traj = tr.exp_trajectory_eul_myobj(traj_original,GH_seq)
        q0_t0 = traj_original[:,0][:3]

        if include_activation_dynamics:
            state_symbols = tuple(q+u+activations)
            specified_symbols = tuple(excitations)
        else:
            state_symbols = tuple(q+u)
            specified_symbols = tuple(activations)

        num_states = len(state_symbols)
        num_q = len(q)
        num_u = len(u)
        num_faux = 0

        num_inputs = len(specified_symbols)
        t = me.dynamicsymbols._t
        objective_traj,objective_traj_jac, objective_traj_t0, objective_traj_t0_jac = eq.custom_objective_eul(len(q),interval_value, GH_seq = GH_seq)
        obj_min_diff,obj_min_diff_jac = eq.objective_activation_diff(num_nodes,interval_value)
        w_diff_act = 0
        w_diff_exc = 1e-3
        w_diff_vel = 1e-3

        def obj(free):
            min_traj = traj_w * np.sum(objective_traj(np.split(free[:num_q*num_nodes],num_q),traj))
            min_SC_t0 = traj_w * np.sum(objective_traj_t0(free[0::num_nodes][:3],q0_t0))

            min_vel_dif = w_diff_vel * np.sum((obj_min_diff(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))))
            min_torque = act_w * interval_value * np.sum(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes]**2)
            if include_activation_dynamics:
                min_exc_dif = w_diff_exc * np.sum((obj_min_diff(np.transpose(np.split(free[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes],num_inputs)))))

            obj = (min_traj + min_torque + min_vel_dif + min_SC_t0) # 

            if include_activation_dynamics:
                obj += min_exc_dif
                
            return obj.item()

        def obj_grad(free):
            grad = np.zeros_like(free)
            grad[:num_q*num_nodes] = traj_w * np.concatenate(objective_traj_jac(np.split(free[:num_q*num_nodes],num_q),traj))
            grad[0::num_nodes][:3] += traj_w * np.sum(objective_traj_t0_jac(free[0::num_nodes][:3],q0_t0))

            grad[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes] += act_w * 2.0 * interval_value * free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes] #+ w_diff_act * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_inputs)*num_nodes],num_inputs)))[0,:,:])))
            if include_activation_dynamics:
                grad[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes] += w_diff_exc * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_q + num_u + num_inputs)*num_nodes:(num_q + num_u + 2*num_inputs)*num_nodes],num_inputs)))[0,:,:])))

            grad[num_q*num_nodes:(num_q + num_u)*num_nodes] += w_diff_vel * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))[0,:,:])))

            return grad
        
        print('obj_check', obj(np.ones(num_states*num_nodes + num_inputs*num_nodes)*0.01))
        print('obj_grad_check', sum(obj_grad(np.ones(num_states*num_nodes + num_inputs*num_nodes)*0.01)))
        instance_constraints = []
        instance_constraints.append(state_symbols[-1].func(0.0)-0)

        bounds1 = (0.0,1.0)
        bounds = (bounds1,)*len(activations)
        bndrs = dict(zip(activations,bounds))
        if include_activation_dynamics:
            bndrs_exc = dict(zip(excitations,(bounds1,)*len(excitations)))
            bndrs.update(bndrs_exc)

        for i in range(num_q):
            if i == 7 or i == 9:
                bndrs.update({q[i]: (min(traj_original[i,:])-0.5, max(traj_original[i,:])+0.5)})
            else:
                bndrs.update({q[i]: (min(traj_original[i,:])-0.2, max(traj_original[i,:])+0.2)})


        start = tm.time()
        prob = Problem(obj, obj_grad, eoms_implicit, state_symbols,
                    num_nodes, interval_value,
                    known_parameter_map={},
                    instance_constraints=instance_constraints,
                    bounds=bndrs,
                    integration_method='midpoint',
        ) #               
        time_to_create = tm.time() - start
        print(time_to_create)
        prob.add_option('max_iter',3000)
        prob.add_option('limited_memory_max_history', 40)
        reload(tr);
        initial_guess = np.zeros(prob.num_free)
        initial_guess[:10*num_nodes] = traj_original.flatten()
        initial_guess[num_q * num_nodes : (num_q + num_u) * num_nodes] = velocity.flatten()


        time_2_solve_start = tm.time()
        solution, info = prob.solve(initial_guess)
        time_2_solve = tm.time() - time_2_solve_start
        print(info['status_msg'])
        print(info['obj_val'])
        act_obj = np.sum(solution[num_states*num_nodes:(num_states + num_inputs)*num_nodes]**2)
        objective_value = prob.obj_value
        print('Objective activations: ', act_obj)
        reload(tr)

        file_name = '../Motions/'+participant[ipar]+'/'+motion_list[imot]+'/' + struct_name + '.mat'
        tr.sol2struct(solution,activations,num_q,num_u,num_faux,num_inputs,num_nodes,time,objective_value,time_2_solve,file_name,include_activation_dynamics)

        file_name_mot = '../Motions/'+participant[ipar]+'/'+motion_list[imot]+'/' + struct_name + '.mot'
        tr.sol2mot_eul(solution, num_nodes, len(q), time, file_name_mot,GH_seq)

dimensions: [0.138021  0.184428  0.0746393]
trans [ 0.         -0.13466143  0.05498011]
equations created
[ 0.1181 -0.0114 -0.01  ]
[-0.004   -0.02226 -0.03024]
79
obj_check 520.5816418584325
obj_grad_check -1624.2954841507626
896.2079684734344
This is Ipopt version 3.14.19, running with linear solver MUMPS 5.7.3.

Number of nonzeros in equality constraint Jacobian...:  8055601
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        0

Total number of variables............................:    27674
                     variables with only lower bounds:        0
                variables with lower and upper bounds:    26664
                     variables with only upper bounds:        0
Total number of equality constraints.................:    14701
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and u

KeyboardInterrupt: 